# 🎙️ VoiceScript AI Engineering Assessment
## Audio Analysis Agent — ffmpeg + LLM Pipeline

**Author:** Dhany Delio Achmad  
**Stack:** Python · ffmpeg/ffprobe · LangChain · Groq · Pydantic

### Architecture
```
audio_file.wav
      ↓
Tool 1: get_audio_metadata()     → ffprobe (duration, bitrate, sample rate, channels)
Tool 2: detect_silence()         → ffmpeg silencedetect filter
Tool 3: detect_clipping()        → ffmpeg volumedetect filter
Tool 4: detect_noise_level()     → ffmpeg astats filter
      ↓
LLM Agent (LangChain + Groq)
      ↓
Pydantic Structured Output → JSON Report
```

## 1. Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install langchain langchain-groq pydantic python-dotenv groq tenacity httpx openai-whisper fastmcp nest_asyncio -q


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


## 2. Setup & Imports

In [2]:
import subprocess
import json
import os
import re
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# LangChain + Groq
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

import nest_asyncio
nest_asyncio.apply()  # Allow asyncio.gather inside Jupyter event loop

load_dotenv()

# Config
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "your_groq_key_here")
# Or use OpenAI if they gave you a key:
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your_openai_key_here")

print("✅ Imports loaded")
print(f"✅ Groq API key: {'set' if GROQ_API_KEY != 'your_groq_key_here' else 'NOT SET'}")

✅ Imports loaded
✅ Groq API key: set


## 3. Pydantic Schema — Structured Output

In [3]:
class AudioQuality(BaseModel):
    silence_ratio: float = Field(default=0.0, description="Ratio of silence to total duration")
    clipping_detected: bool = Field(default=False, description="Whether clipping was detected")
    avg_volume_db: Optional[float] = Field(default=None, description="Average volume in dB")

class AudioAnalysisReport(BaseModel):
    file_name: str
    duration_seconds: float = Field(default=0.0, description="Total duration of the audio file in seconds")
    audio_quality: AudioQuality
    issues: List[str] = Field(default=[], description="List of detected issues")
    llm_summary: str = Field(default="", description="LLM-generated human-readable summary")
    recommendations: List[str] = Field(default=[], description="LLM-generated recommendations")
    overall_usability: str = Field(default="unknown", description="usable/partially_usable/unusable")
    detected_languages: List[str] = Field(default=[], description="Languages identified in the audio, e.g., ['id', 'en']")

# Internal metadata model — used during analysis but not part of final report
class AudioMetadata(BaseModel):
    duration_seconds: float = Field(default=0.0)
    bitrate_kbps: Optional[float] = Field(default=None)
    sample_rate_hz: Optional[int] = Field(default=None)
    channels: Optional[int] = Field(default=None)
    codec: Optional[str] = Field(default=None)
    file_size_mb: Optional[float] = Field(default=None)

print("✅ Pydantic schemas defined")

✅ Pydantic schemas defined


## 4. Tool 1 — Audio Metadata Extractor (ffprobe)

In [4]:
def get_audio_metadata(file_path: str) -> AudioMetadata:
    """
    Extract audio metadata using ffprobe.
    Returns structured AudioMetadata.
    """
    try:
        cmd = [
            "ffprobe",
            "-v", "quiet",
            "-print_format", "json",
            "-show_format",
            "-show_streams",
            file_path
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        data = json.loads(result.stdout)

        # Get audio stream
        audio_stream = next(
            (s for s in data.get("streams", []) if s.get("codec_type") == "audio"),
            {}
        )
        fmt = data.get("format", {})

        duration = float(fmt.get("duration", 0))
        bitrate = float(fmt.get("bit_rate", 0)) / 1000 if fmt.get("bit_rate") else None
        file_size = round(float(fmt.get("size", 0)) / (1024 * 1024), 2) if fmt.get("size") else None

        return AudioMetadata(
            duration_seconds=duration,
            bitrate_kbps=bitrate,
            sample_rate_hz=int(audio_stream.get("sample_rate", 0)) or None,
            channels=audio_stream.get("channels"),
            codec=audio_stream.get("codec_name"),
            file_size_mb=file_size
        )

    except Exception as e:
        print(f"❌ Metadata extraction failed: {e}")
        return AudioMetadata()


print("✅ Tool 1: get_audio_metadata() ready")

✅ Tool 1: get_audio_metadata() ready


## 5. Tool 2 — Silence Detection (ffmpeg)

In [5]:
def detect_silence(file_path: str, silence_thresh_db: float = -40.0, 
                   min_silence_duration: float = 2.0) -> dict:
    """
    Detect silence segments using ffmpeg silencedetect filter.
    
    Args:
        silence_thresh_db: Volume threshold for silence (default -40dB)
        min_silence_duration: Minimum silence duration in seconds (default 2s)
    """
    try:
        cmd = [
            "ffmpeg", "-i", file_path,
            "-af", f"silencedetect=noise={silence_thresh_db}dB:d={min_silence_duration}",
            "-f", "null", "-"
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        stderr = result.stderr

        # Parse silence segments
        silence_starts = re.findall(r"silence_start: ([\d.]+)", stderr)
        silence_ends = re.findall(r"silence_end: ([\d.]+)", stderr)
        silence_durations = re.findall(r"silence_duration: ([\d.]+)", stderr)

        segments = []
        for i, (start, end, dur) in enumerate(
            zip(silence_starts, silence_ends, silence_durations)
        ):
            segments.append({
                "start": float(start),
                "end": float(end),
                "duration": float(dur),
                "label": f"Silence {i+1}: {float(start):.1f}s – {float(end):.1f}s ({float(dur):.1f}s)"
            })

        total_silence = sum(float(d) for d in silence_durations)

        return {
            "segments": segments,
            "total_silence_seconds": total_silence,
            "segment_count": len(segments)
        }

    except Exception as e:
        print(f"❌ Silence detection failed: {e}")
        return {"segments": [], "total_silence_seconds": 0, "segment_count": 0}


print("✅ Tool 2: detect_silence() ready")

✅ Tool 2: detect_silence() ready


## 6. Tool 3 — Volume & Clipping Detection (ffmpeg)

In [6]:
def detect_volume_and_clipping(file_path: str) -> dict:
    """
    Detect volume levels and clipping using ffmpeg volumedetect.
    Clipping = max_volume near 0dB.
    """
    try:
        cmd = [
            "ffmpeg", "-i", file_path,
            "-af", "volumedetect",
            "-f", "null", "-"
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        stderr = result.stderr

        # Parse volume stats
        mean_volume = re.search(r"mean_volume: ([\-\d.]+) dB", stderr)
        max_volume = re.search(r"max_volume: ([\-\d.]+) dB", stderr)

        avg_db = float(mean_volume.group(1)) if mean_volume else None
        max_db = float(max_volume.group(1)) if max_volume else None

        # Clipping = max volume >= -1dB
        clipping = max_db is not None and max_db >= -1.0

        # Noise level classification
        if avg_db is None:
            noise_level = "unknown"
        elif avg_db > -20:
            noise_level = "high"
        elif avg_db > -35:
            noise_level = "medium"
        else:
            noise_level = "low"

        return {
            "avg_volume_db": avg_db,
            "max_volume_db": max_db,
            "clipping_detected": clipping,
            "noise_level": noise_level
        }

    except Exception as e:
        print(f"❌ Volume detection failed: {e}")
        return {
            "avg_volume_db": None, 
            "max_volume_db": None,
            "clipping_detected": False,
            "noise_level": "unknown"
        }


print("✅ Tool 3: detect_volume_and_clipping() ready")

✅ Tool 3: detect_volume_and_clipping() ready


## 6b. Tool 4 — Language Detection (Whisper)

In [7]:
import whisper as _whisper
import tempfile
import os as _os

# Load Whisper tiny model once — fast, ~75MB, accurate for language ID
_whisper_model = None

def _get_whisper_model():
    global _whisper_model
    if _whisper_model is None:
        print("   🔄 Loading Whisper tiny model (first run only)...")
        _whisper_model = _whisper.load_model("tiny")
    return _whisper_model


def detect_language_whisper(file_path: str, sample_duration: int = 30) -> dict:
    """
    Tool 4: Detect spoken language(s) using OpenAI Whisper.

    Extracts the first `sample_duration` seconds via ffmpeg,
    runs Whisper language detection (no full transcription needed),
    and returns the detected language code + confidence.

    Args:
        file_path       : path to audio file
        sample_duration : seconds to sample from start (default 30s)

    Returns:
        {
            'language'   : ISO 639-1 code, e.g. 'en', 'id'
            'confidence' : probability 0.0–1.0
            'method'     : 'whisper'
        }
    """
    try:
        # Extract audio sample to a temp WAV file (Whisper works best with WAV)
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp_path = tmp.name

        cmd = [
            "ffmpeg", "-y", "-i", file_path,
            "-t", str(sample_duration),   # first N seconds only
            "-ar", "16000",               # Whisper expects 16kHz
            "-ac", "1",                   # mono
            "-f", "wav", tmp_path
        ]
        subprocess.run(cmd, capture_output=True, timeout=30, check=True)

        # Load audio and run Whisper language detection
        model = _get_whisper_model()
        audio = _whisper.load_audio(tmp_path)
        audio = _whisper.pad_or_trim(audio)

        # Compute log-Mel spectrogram and detect language
        mel = _whisper.log_mel_spectrogram(audio).to(model.device)
        _, probs = model.detect_language(mel)

        # Top language
        top_lang = max(probs, key=probs.get)
        top_conf = float(probs[top_lang])

        # Check for bilingual: if 2nd language prob > 15%, flag as bilingual
        sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)
        detected = [top_lang]
        if len(sorted_probs) > 1 and sorted_probs[1][1] > 0.15:
            detected.append(sorted_probs[1][0])

        return {
            "language": top_lang,
            "languages": detected,
            "confidence": round(top_conf, 3),
            "method": "whisper"
        }

    except Exception as e:
        print(f"   ⚠️  Whisper language detection failed: {e}")
        return {"language": "unknown", "languages": ["unknown"], "confidence": 0.0, "method": "failed"}

    finally:
        # Clean up temp file
        try:
            _os.unlink(tmp_path)
        except Exception:
            pass


print("✅ Tool 4: detect_language_whisper() ready")


✅ Tool 4: detect_language_whisper() ready


## 7. Multi-Agent LLM Pipeline (Acoustic · Linguistic · Manager)

In [8]:
# ── Fault Tolerance imports ───────────────────────────
from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
    before_sleep_log,
)
import logging
from groq import RateLimitError
import httpx  # Timeout comes from httpx when using Groq SDK

logging.basicConfig(level=logging.WARNING)
_logger = logging.getLogger("voicescript")


# ── Shared helpers ────────────────────────────────────────
def _parse_llm_json(content: str) -> dict:
    """Strip markdown fences and parse JSON from LLM response."""
    content = content.strip()
    if "```json" in content:
        content = content.split("```json")[1].split("```")[0].strip()
    elif "```" in content:
        content = content.split("```")[1].split("```")[0].strip()
    return json.loads(content)


def _make_llm(model: str = "llama-3.3-70b-versatile") -> ChatGroq:
    """Shared factory — model is configurable per agent."""
    return ChatGroq(
        api_key=GROQ_API_KEY,
        model=model,
        temperature=0.3,
        request_timeout=30,
    )


# Model assignment per agent
# Agent 1 & 2 : llama-3.1-8b-instant  → fast, cheap, structured JSON tasks
# Agent 3      : llama-3.3-70b-versatile → strong reasoning for final synthesis
MODEL_SPECIALIST = "llama-3.1-8b-instant"
MODEL_MANAGER    = "llama-3.3-70b-versatile"


# ── Retry decorator (Circuit Breaker) ────────────────────
# Retries on RateLimitError or network Timeout.
# Strategy: exponential backoff 2s → 4s → 8s, max 3 attempts.
_llm_retry = retry(
    retry=retry_if_exception_type((RateLimitError, httpx.TimeoutException)),
    wait=wait_exponential(multiplier=1, min=2, max=8),
    stop=stop_after_attempt(3),
    before_sleep=before_sleep_log(_logger, logging.WARNING),
    reraise=True,   # re-raise final exception so per-agent fallback catches it
)


@_llm_retry
def _invoke_with_retry(messages: list, model: str = MODEL_MANAGER) -> str:
    """Single retryable LLM call. Returns raw response content string."""
    response = _make_llm(model=model).invoke(messages)
    return response.content


# ──────────────────────────────────────────────────────────
# AGENT 1 — ACOUSTIC QUALITY EXPERT
# Input : volume_data, silence_data, metadata
# Output: overall_usability, acoustic_issues
# ──────────────────────────────────────────────────────────
def agent_acoustic_expert(
    metadata: "AudioMetadata",
    silence_data: dict,
    volume_data: dict
) -> dict:
    """
    Agent 1: Acoustic Quality Expert.
    Focuses strictly on dB levels, clipping, silence, and noise.
    Returns overall_usability verdict and acoustic_issues list.
    Fault-tolerant: retries on RateLimitError/Timeout, falls back to
    static rule-based verdict if Groq is completely unavailable.
    """
    silence_ratio = (
        silence_data["total_silence_seconds"] / metadata.duration_seconds
        if metadata.duration_seconds > 0 else 0
    )
    context = f"""ACOUSTIC DATA FOR ANALYSIS:

Volume Metrics:
- Average volume  : {volume_data['avg_volume_db']} dB
- Max volume      : {volume_data['max_volume_db']} dB
- Clipping        : {volume_data['clipping_detected']}
- Noise level     : {volume_data['noise_level']}

Silence Metrics:
- Silence ratio   : {silence_ratio:.2%}
- Silence segments: {silence_data['segment_count']}
- Segments detail : {json.dumps([s['label'] for s in silence_data['segments'][:5]])}

Technical Specs:
- Bitrate  : {metadata.bitrate_kbps} kbps
- Codec    : {metadata.codec}
- Channels : {metadata.channels}
"""
    messages = [
        SystemMessage(content="""You are an ACOUSTIC QUALITY EXPERT for legal audio recordings.
Your job is to coldly evaluate the acoustic data and output a machine-readable verdict.

RULES:
- Focus ONLY on acoustic metrics: dB levels, clipping, silence, noise, bitrate.
- List every acoustic problem you find in acoustic_issues (be specific, use dB values).
- Apply this STRICT usability classification:
  * 'usable'           : noise_level 'low', no clipping, silence_ratio < 20%, bitrate >= 64kbps
  * 'partially_usable' : noise_level 'medium' OR silence_ratio 20-40% OR bitrate 32-63kbps
  * 'unusable'         : clipping AND noise_level 'high', OR silence_ratio > 40%, OR bitrate < 32kbps

MATH VALIDATION — MANDATORY before writing any issue:
- Verify every numeric comparison explicitly before writing it as an issue.
- Bitrate rule  : only flag as low if bitrate value is STRICTLY LESS THAN 64. If bitrate >= 64, do NOT mention it as an issue.
- Silence rule  : only flag if silence_ratio is STRICTLY GREATER THAN 0.20 (20%). If silence_ratio <= 0.20, do NOT write any issue about silence ratio exceeding 20%. Example: silence_ratio=0.1341 is 13.41% which is BELOW 20% — do NOT flag it.
- Volume rule   : only flag avg_volume_db as low if value is STRICTLY LESS THAN -40 dB.
- STRICT THRESHOLD CHECK: Before writing any issue text, re-read the exact raw number from the input and compare it against the threshold numerically. If the number does NOT breach the threshold, you are FORBIDDEN from writing that issue. Do not approximate, round, or misread values.
- When in doubt, recalculate: re-read the exact number from the input before comparing.

Respond ONLY with valid JSON:
{
  \"overall_usability\": \"usable|partially_usable|unusable\",
  \"acoustic_issues\": [\"issue 1\", \"issue 2\"]
}"""),
        HumanMessage(content=context)
    ]

    try:
        raw = _invoke_with_retry(messages, model=MODEL_SPECIALIST)
        result = _parse_llm_json(raw)
        print(f"   🔊 Acoustic verdict : {result.get('overall_usability', '?').upper()} | "
              f"Issues: {len(result.get('acoustic_issues', []))}")
        return result

    except Exception as e:
        # ── FALLBACK: static rule-based verdict ──────────
        print(f"   ⚠️  Acoustic Expert LLM unavailable ({type(e).__name__}). Using static fallback.")
        fallback_issues = []
        if volume_data["clipping_detected"]:
            fallback_issues.append(f"[FALLBACK] Audio clipping detected (max {volume_data['max_volume_db']} dB)")
        if volume_data["noise_level"] in ("medium", "high"):
            fallback_issues.append(f"[FALLBACK] Noise level is {volume_data['noise_level']} (avg {volume_data['avg_volume_db']} dB)")
        if silence_ratio > 0.20:
            fallback_issues.append(f"[FALLBACK] High silence ratio: {silence_ratio:.1%}")
        if metadata.bitrate_kbps and metadata.bitrate_kbps < 64:
            fallback_issues.append(f"[FALLBACK] Low bitrate: {metadata.bitrate_kbps:.0f} kbps")

        # Rule-based usability verdict
        if volume_data["clipping_detected"] and volume_data["noise_level"] == "high":
            fallback_usability = "unusable"
        elif volume_data["noise_level"] in ("medium", "high") or silence_ratio > 0.20:
            fallback_usability = "partially_usable"
        else:
            fallback_usability = "usable"

        return {
            "overall_usability": fallback_usability,
            "acoustic_issues": fallback_issues if fallback_issues
                else ["[FALLBACK] LLM unavailable — manual acoustic review required"]
        }


# ──────────────────────────────────────────────────────────
# AGENT 2 — LINGUISTIC & LANGUAGE EXPERT
# Input : file_name, metadata, silence_data
# Output: detected_languages, linguistic_issues
# ──────────────────────────────────────────────────────────
def agent_linguistic_expert(
    file_name: str,
    metadata: "AudioMetadata",
    silence_data: dict,
    whisper_result: dict = None
) -> dict:
    """
    Agent 2: Linguistic & Language Expert.
    Detects language(s) and code-switching patterns.
    Uses Whisper language detection data (Tool 4) as ground truth when available.
    Fault-tolerant: retries on RateLimitError/Timeout, falls back to
    ["unknown"] if Groq is completely unavailable.
    """
    silence_ratio = (
        silence_data["total_silence_seconds"] / metadata.duration_seconds
        if metadata.duration_seconds > 0 else 0
    )

    # Build Whisper context block if available
    whisper_block = ""
    if whisper_result and whisper_result.get("method") == "whisper":
        whisper_block = f"""
WHISPER LANGUAGE DETECTION (ground truth — from actual audio signal):
- Primary language : {whisper_result['language']} (confidence: {whisper_result['confidence']:.1%})
- All detected     : {whisper_result['languages']}
NOTE: Trust this data over filename inference. Use it as the primary source for detected_languages.
"""

    context = f"""LINGUISTIC CONTEXT FOR ANALYSIS:

File Name   : {file_name}
Duration    : {metadata.duration_seconds:.1f}s ({metadata.duration_seconds / 60:.1f} min)
Sample Rate : {metadata.sample_rate_hz} Hz
Channels    : {metadata.channels}
Silence     : {silence_ratio:.2%} of total duration ({silence_data['segment_count']} segments)
{whisper_block}
Silence segments (for rhythm/pacing analysis):
{json.dumps([s['label'] for s in silence_data['segments'][:5]])}
"""
    messages = [
        SystemMessage(content="""You are a LINGUISTIC & LANGUAGE EXPERT for legal audio recordings.
Your job is to detect language(s) and identify any linguistic risks that could impair transcription.

RULES:
- If WHISPER LANGUAGE DETECTION data is present in the context, use it as the PRIMARY source
  for detected_languages. Do NOT override it with filename guesses.
- If Whisper shows multiple languages with > 15% confidence, flag as bilingual code-switching.
- detected_languages rules:
  * Bilingual Indonesian-English (code-switching)  → [\"id\", \"en\"]
  * Monolingual Indonesian                         → [\"id\"]
  * Monolingual English                            → [\"en\"]
  * Unclear / ambiguous                            → [\"unknown\"]
- List EVERY linguistic risk in linguistic_issues:
  * Code-switching detected (bilingual)
  * Unusual silence patterns suggesting cross-talk
  * File name suggests non-standard content (music, ambient, etc.)
  * Very short/long duration anomalies
- If the file name suggests music or non-speech content, flag it explicitly.

Respond ONLY with valid JSON:
{
  \"detected_languages\": [\"en\"],
  \"linguistic_issues\": [\"issue 1\", \"issue 2\"]
}"""),
        HumanMessage(content=context)
    ]

    try:
        raw = _invoke_with_retry(messages, model=MODEL_SPECIALIST)
        result = _parse_llm_json(raw)
        # Override with Whisper if LLM returned "unknown" but Whisper has a real answer
        if (whisper_result and whisper_result.get("method") == "whisper"
                and result.get("detected_languages") in (["unknown"], [])):
            result["detected_languages"] = whisper_result["languages"]
        print(f"   🗣️  Languages detected: {result.get('detected_languages', ['?'])} "
              f"(whisper: {whisper_result.get('language','?') if whisper_result else 'n/a'}) | "
              f"Linguistic issues: {len(result.get('linguistic_issues', []))}")
        return result

    except Exception as e:
        # ── FALLBACK: use Whisper result directly if available ────
        print(f"   ⚠️  Linguistic Expert LLM unavailable ({type(e).__name__}). Using fallback.")
        if whisper_result and whisper_result.get("method") == "whisper":
            fallback_langs = whisper_result["languages"]
            note = f"[FALLBACK] LLM unavailable — language from Whisper: {whisper_result['language']} ({whisper_result['confidence']:.1%})"
        else:
            name_lower = file_name.lower()
            id_hints  = any(w in name_lower for w in ["rekaman", "sidang", "depo", "wawancara"])
            eng_hints = any(w in name_lower for w in ["record", "deposition", "interview", "meeting"])
            if id_hints and eng_hints:
                fallback_langs = ["id", "en"]
            elif id_hints:
                fallback_langs = ["id"]
            elif eng_hints:
                fallback_langs = ["en"]
            else:
                fallback_langs = ["unknown"]
            note = "[FALLBACK] LLM unavailable — language detection based on filename heuristic only"

        return {
            "detected_languages": fallback_langs,
            "linguistic_issues": [note]
        }


# ──────────────────────────────────────────────────────────
# AGENT 3 — MANAGER / RECONCILER
# Input : all raw data + Agent 1 & Agent 2 results
# Output: summary, recommendations
# ──────────────────────────────────────────────────────────
def agent_manager(
    file_name: str,
    metadata: "AudioMetadata",
    all_issues: List[str],
    acoustic_result: dict,
    linguistic_result: dict
) -> dict:
    """
    Agent 3: The Manager / Reconciler.
    Aggregates Agent 1 & 2 findings into a professional narrative.
    Fault-tolerant: retries on RateLimitError/Timeout, falls back to
    a templated summary built from agent outputs if Groq is unavailable.
    """
    context = f"""AGGREGATED ANALYSIS REPORT FOR: {file_name}

=== AGENT 1 — ACOUSTIC EXPERT FINDINGS ===
Usability Verdict  : {acoustic_result.get('overall_usability', 'unknown').upper()}
Acoustic Issues    :
{chr(10).join(f'  - {i}' for i in acoustic_result.get('acoustic_issues', [])) or '  - None detected'}

=== AGENT 2 — LINGUISTIC EXPERT FINDINGS ===
Detected Languages : {linguistic_result.get('detected_languages', ['unknown'])}
Linguistic Issues  :
{chr(10).join(f'  - {i}' for i in linguistic_result.get('linguistic_issues', [])) or '  - None detected'}

=== INFRASTRUCTURE ISSUES (Python analysis) ===
{chr(10).join(f'  - {i}' for i in all_issues) or '  - None detected'}

=== FILE SPECS ===
Duration : {metadata.duration_seconds:.1f}s ({metadata.duration_seconds / 60:.1f} min)
Bitrate  : {metadata.bitrate_kbps} kbps
Codec    : {metadata.codec} | Channels: {metadata.channels}
"""
    messages = [
        SystemMessage(content="""You are the SENIOR MANAGER and FINAL RECONCILER for an audio quality analysis pipeline.
You have received findings from two specialist agents (Acoustic Expert and Linguistic Expert).
Your job is to synthesize their findings into a polished, professional report for legal teams.

STALWART RULE — FACT-CHECK BEFORE WRITING:
Before including any issue from upstream agents into your summary or recommendations, verify its
mathematical and logical correctness against the FILE SPECS provided in the context.
- If an upstream agent claims bitrate is low but FILE SPECS show bitrate >= 64 kbps, DISCARD that claim.
- If an upstream agent claims silence is high but the ratio is < 20%, DISCARD that claim.
- If an upstream agent claims volume is very low but avg_volume_db > -40 dB, DISCARD that claim.
- Never propagate a factually incorrect statement into the final report, even if an upstream agent said it.
- When in doubt, trust the raw numbers in FILE SPECS over the agent's text interpretation.

SYNTHESIS RULES:
- Write a clear, professional summary (2-4 sentences) integrating BOTH acoustic and linguistic findings.
  Do NOT just repeat bullet points — synthesize into a coherent narrative.
- Provide 4-6 specific, prioritized, actionable recommendations ordered from most critical to least.
  Each recommendation must be a complete sentence with a clear action verb.
- If both agents agree on severity, amplify that conclusion.
- If agents found conflicting signals, acknowledge the nuance in your summary.

Respond ONLY with valid JSON:
{
  \"summary\": \"...\",
  \"recommendations\": [\"...\", \"...\"]
}"""),
        HumanMessage(content=context)
    ]

    try:
        raw = _invoke_with_retry(messages, model=MODEL_MANAGER)
        result = _parse_llm_json(raw)
        print(f"   📝 Manager summary generated ({len(result.get('recommendations', []))} recommendations)")
        return result

    except Exception as e:
        # ── FALLBACK: template-based summary from agent outputs ──
        print(f"   ⚠️  Manager Agent LLM unavailable ({type(e).__name__}). Using static fallback summary.")
        usability    = acoustic_result.get('overall_usability', 'unknown')
        langs        = linguistic_result.get('detected_languages', ['unknown'])
        n_issues     = len(all_issues)
        fallback_summary = (
            f"[FALLBACK REPORT] Audio file '{file_name}' was assessed as '{usability}' "
            f"by the Acoustic Expert. "
            f"Detected language(s): {', '.join(langs)}. "
            f"A total of {n_issues} issue(s) were identified across acoustic, linguistic, "
            f"and infrastructure checks. LLM Manager was unavailable — manual review recommended."
        )
        fallback_recs = [
            "Perform a manual listening review to verify all flagged acoustic issues.",
            "Apply noise reduction pre-processing before submitting for legal transcription.",
            "Re-run this pipeline once Groq API connectivity is restored for full LLM analysis.",
        ]
        if n_issues > 0:
            fallback_recs.insert(0, f"Address the {n_issues} flagged issue(s) before proceeding with transcription.")
        return {
            "summary": fallback_summary,
            "recommendations": fallback_recs
        }


print("✅ Multi-Agent LLM pipeline ready (with Fault Tolerance):")
print(f"   🔊 Agent 1 — Acoustic Expert      → {MODEL_SPECIALIST}")
print(f"   🗣️  Agent 2 — Linguistic Expert    → {MODEL_SPECIALIST}")
print(f"   📋 Agent 3 — Manager / Reconciler → {MODEL_MANAGER}")
print("   🔁 Retry  — tenacity (RateLimitError + Timeout, max 3 attempts, exp backoff 2-8s)")
print("   🛡️  Fallback — static rule-based response per agent if all retries exhausted")

✅ Multi-Agent LLM pipeline ready (with Fault Tolerance):
   🔊 Agent 1 — Acoustic Expert      → llama-3.1-8b-instant
   🗣️  Agent 2 — Linguistic Expert    → llama-3.1-8b-instant
   📋 Agent 3 — Manager / Reconciler → llama-3.3-70b-versatile
   🔁 Retry  — tenacity (RateLimitError + Timeout, max 3 attempts, exp backoff 2-8s)
   🛡️  Fallback — static rule-based response per agent if all retries exhausted


## 8. Main Orchestrator — Multi-Agent Full Pipeline

In [9]:
def analyze_audio(file_path: str) -> AudioAnalysisReport:
    """
    Main orchestrator — Multi-Agent pipeline.

    Pipeline:
    1. Extract metadata          (ffprobe)
    2. Detect silence            (ffmpeg silencedetect)
    3. Detect volume + clipping  (ffmpeg volumedetect)
    4. Collect infra issues      (Python rules)
    5. Agent 1 — Acoustic Expert (LLM)
    6. Agent 2 — Linguistic Expert (LLM)
    7. Merge all issues
    8. Agent 3 — Manager/Reconciler (LLM)
    9. Build structured AudioAnalysisReport
    """
    file_name = Path(file_path).name
    print(f"\n🎙️ Analyzing: {file_name}")
    print("=" * 55)

    # ── Step 1: Metadata ──────────────────────────────────
    print("📊 Step 1: Extracting metadata...")
    metadata = get_audio_metadata(file_path)
    print(f"   Duration: {metadata.duration_seconds:.1f}s | "
          f"Codec: {metadata.codec} | "
          f"Channels: {metadata.channels} | "
          f"Bitrate: {metadata.bitrate_kbps} kbps")

    # ── Step 2: Silence ───────────────────────────────────
    print("🔇 Step 2: Detecting silence...")
    silence_data = detect_silence(file_path)
    silence_ratio = (
        silence_data["total_silence_seconds"] / metadata.duration_seconds
        if metadata.duration_seconds > 0 else 0
    )
    print(f"   Segments: {silence_data['segment_count']} | "
          f"Total: {silence_data['total_silence_seconds']:.1f}s | "
          f"Ratio: {silence_ratio:.1%}")

    # ── Step 3: Volume + Clipping ─────────────────────────
    print("📢 Step 3: Analyzing volume and clipping...")
    volume_data = detect_volume_and_clipping(file_path)
    print(f"   Avg: {volume_data['avg_volume_db']} dB | "
          f"Max: {volume_data['max_volume_db']} dB | "
          f"Clipping: {volume_data['clipping_detected']} | "
          f"Noise: {volume_data['noise_level']}")

    # ── Step 4: Infrastructure issue rules ───────────────
    print("⚠️  Step 4: Collecting infrastructure issues...")
    infra_issues = []

    if silence_ratio > 0.20:
        infra_issues.append(f"High silence ratio: {silence_ratio:.1%} of recording is silent")
    if silence_data["segment_count"] > 5:
        infra_issues.append(f"Multiple silence segments: {silence_data['segment_count']} segments detected")
    for seg in silence_data["segments"]:
        if seg["duration"] > 30:
            infra_issues.append(f"Extended silence: {seg['label']}")
    if volume_data["clipping_detected"]:
        infra_issues.append("Audio clipping detected — may cause distortion")
    if volume_data["avg_volume_db"] and volume_data["avg_volume_db"] < -40:
        infra_issues.append(f"Very low average volume: {volume_data['avg_volume_db']} dB")
    if volume_data["noise_level"] in ("medium", "high"):
        label = "Medium" if volume_data["noise_level"] == "medium" else "High"
        infra_issues.append(f"{label} background noise detected — may reduce transcription accuracy")
    if metadata.bitrate_kbps and metadata.bitrate_kbps < 64:
        infra_issues.append(f"Low bitrate: {metadata.bitrate_kbps:.0f} kbps — may affect audio quality")

    print(f"   Infra issues: {len(infra_issues)}")
    for issue in infra_issues:
        print(f"   ⚠️  {issue}")

    # ── Step 5+6: Whisper + Agent 1 + Agent 2 — parallel ─
    # Whisper runs first (needed by Agent 2), then both agents
    # fire concurrently via asyncio.gather — same token cost,
    # ~50% faster since Agent 1 & 2 are fully independent.
    import asyncio

    print("\n🎙️  Step 5: Tool 4 — Whisper language detection...")
    whisper_result = detect_language_whisper(file_path)
    print(f"   Whisper: {whisper_result['language']} "
          f"(conf: {whisper_result['confidence']:.1%}) | "
          f"method: {whisper_result['method']}")

    print("\n⚡ Step 6: Agent 1 & Agent 2 running in parallel...")
    async def _run_agents_parallel():
        return await asyncio.gather(
            asyncio.to_thread(agent_acoustic_expert, metadata, silence_data, volume_data),
            asyncio.to_thread(
                agent_linguistic_expert,
                file_name, metadata, silence_data,
                whisper_result
            ),
        )

    acoustic_result, linguistic_result = asyncio.run(_run_agents_parallel())

    # ── Step 7: Merge + sanitize all issues ──────────────
    print("\n🔗 Step 7: Merging and sanitizing issues...")

    # Keyword map: if an LLM issue shares a keyword with a Python infra issue,
    # the Python version is authoritative — drop the LLM duplicate.
    KEYWORD_MAP = {
        "clipping": ["clipping", "clip"],
        "noise":    ["noise", "background"],
        "bitrate":  ["bitrate", "kbps"],
        "silence":  ["silence"],
        "volume":   ["volume", "low volume"],
    }

    def _topic_keys(text: str) -> set:
        """Return the set of topic keys that match anywhere in the text."""
        txt = text.lower()
        return {
            topic
            for topic, keywords in KEYWORD_MAP.items()
            if any(kw in txt for kw in keywords)
        }

    def _sanitize_and_deduplicate(
        infra: list, llm_acoustic: list, llm_linguistic: list,
        metadata: AudioMetadata, silence_ratio: float, volume_data: dict
    ) -> list:
        """
        Two-pass filter applied to LLM-generated issues before merging:

        Pass 1 — Math hallucination guard (same rules as before):
          Drop any LLM issue that contradicts raw measured numbers.

        Pass 2 — Semantic deduplication:
          If an LLM issue covers the same topic as an existing Python infra
          issue, the Python version is authoritative → drop the LLM duplicate.
          LLM-only insights (no Python equivalent) are kept as genuine value.
        """
        # Build topic coverage of Python infra issues
        infra_topics: set = set()
        for issue in infra:
            infra_topics |= _topic_keys(issue)

        filtered, dropped = [], []

        for issue in llm_acoustic + llm_linguistic:
            txt = issue.lower()
            discard = False

            # ── Pass 1: math / hallucination guard ──────────
            # False silence > 20% claim
            if silence_ratio <= 0.20 and "silence" in txt and (
                "exceed" in txt or "above" in txt or "over" in txt
                or "greater" in txt or "exceeds 20" in txt
                or ("20%" in txt and "strictly less than 20" not in txt)
            ):
                discard = True

            # False low-bitrate claim when bitrate >= 64
            if (
                not discard
                and metadata.bitrate_kbps is not None
                and metadata.bitrate_kbps >= 64
                and "bitrate" in txt
                and any(w in txt for w in ["low", "below", "less than", "under", "64"])
            ):
                discard = True

            # Prompt template placeholders leaked
            if not discard and txt.strip() in ("issue 1", "issue 2", "issue 3", "issue 4"):
                discard = True

            # Raw data dumps — not actionable issues
            if not discard and (
                txt.startswith("noise_level")
                or txt.startswith("noise level")
                or txt.startswith("silence_segments")
                or txt.startswith("silence_ratio")
                or "strictly less than 20" in txt
                or "is below 20" in txt
            ):
                discard = True

            # ── Pass 2: semantic deduplication ──────────────
            if not discard:
                llm_topics = _topic_keys(issue)
                if llm_topics & infra_topics:
                    # Overlaps with a Python-generated issue → drop duplicate
                    discard = True

            if discard:
                dropped.append(issue)
            else:
                filtered.append(issue)

        if dropped:
            print(f"   🧹 Sanitizer dropped {len(dropped)} issue(s) "
                  f"(hallucinations + duplicates):")
            for d in dropped:
                print(f"      ✂️  {d}")

        # Python infra issues first (authoritative), then unique LLM insights
        return list(dict.fromkeys(infra + filtered))

    all_issues = _sanitize_and_deduplicate(
        infra_issues,
        acoustic_result.get("acoustic_issues", []),
        linguistic_result.get("linguistic_issues", []),
        metadata, silence_ratio, volume_data
    )
    print(f"   Total issues: {len(all_issues)} "
          f"(infra: {len(infra_issues)} | "
          f"llm unique: {len(all_issues) - len(infra_issues)})"
    )

    # ── Step 8: Agent 3 — Manager / Reconciler ───────────
    print("\n📋 Step 8: Agent 3 — Manager synthesizing final report...")
    manager_result = agent_manager(
        file_name, metadata, all_issues, acoustic_result, linguistic_result
    )

    # ── Step 9: Build Pydantic report (schema matches assessment spec) ──
    report = AudioAnalysisReport(
        file_name=file_name,
        duration_seconds=round(metadata.duration_seconds, 2),
        audio_quality=AudioQuality(
            silence_ratio=round(silence_ratio, 4),
            clipping_detected=volume_data["clipping_detected"],
            avg_volume_db=volume_data["avg_volume_db"],
        ),
        issues=all_issues,
        llm_summary=manager_result.get("summary", ""),
        recommendations=manager_result.get("recommendations", []),
        overall_usability=acoustic_result.get("overall_usability", "unknown"),
        detected_languages=linguistic_result.get("detected_languages", ["unknown"])
    )

    print(f"\n{'=' * 55}")
    print(f"✅ Analysis complete — {file_name}")
    print(f"   Usability  : {report.overall_usability.upper()}")
    print(f"   Languages  : {report.detected_languages}")
    print(f"   Issues     : {len(report.issues)} total")
    return report


print("✅ Main orchestrator: analyze_audio() ready (multi-agent)")

✅ Main orchestrator: analyze_audio() ready (multi-agent)


## 9. Run Analysis — Both Audio Files

In [10]:
# ============================================================
# Analyze both audio files in this folder
# ============================================================
import os

AUDIO_FILES = [
    "audio_samples/bad_audio.mp3",
    "audio_samples/moonlight-plaza.mp3",
]

all_reports = []

for audio_file in AUDIO_FILES:
    if not os.path.exists(audio_file):
        print(f"\n⚠️  File not found: {audio_file} — skip")
        continue

    report = analyze_audio(audio_file)
    all_reports.append(report)

    print("\n" + "=" * 50)
    print(f"📋 JSON REPORT — {report.file_name}")
    print("=" * 50)
    print(json.dumps(report.model_dump(), indent=2))
    print()



🎙️ Analyzing: bad_audio.mp3
📊 Step 1: Extracting metadata...
   Duration: 121.1s | Codec: mp3 | Channels: 1 | Bitrate: 47.235 kbps
🔇 Step 2: Detecting silence...
   Segments: 4 | Total: 16.2s | Ratio: 13.4%
📢 Step 3: Analyzing volume and clipping...
   Avg: -16.6 dB | Max: -0.4 dB | Clipping: True | Noise: high
⚠️  Step 4: Collecting infrastructure issues...
   Infra issues: 3
   ⚠️  Audio clipping detected — may cause distortion
   ⚠️  High background noise detected — may reduce transcription accuracy
   ⚠️  Low bitrate: 47 kbps — may affect audio quality

🎙️  Step 5: Tool 4 — Whisper language detection...
   🔄 Loading Whisper tiny model (first run only)...
   Whisper: en (conf: 98.8%) | method: whisper

⚡ Step 6: Agent 1 & Agent 2 running in parallel...
   🗣️  Languages detected: ['en'] (whisper: en) | Linguistic issues: 0
   🔊 Acoustic verdict : UNUSABLE | Issues: 4

🔗 Step 7: Merging and sanitizing issues...
   🧹 Sanitizer dropped 4 issue(s) (hallucinations + duplicates):
      ✂️

## 10. Side-by-Side Comparison — Summary

In [11]:
# ============================================================
# Side-by-side comparison summary of both analyzed files
# ============================================================
print("\n" + "=" * 60)
print("📊 ANALYSIS RESULTS COMPARISON")
print("=" * 60)

headers = ["File", "Duration", "Avg dB", "Max dB", "Clipping", "Noise", "Silence Ratio", "Issues", "Usability"]
print(f"{'File':<25} {'Duration':>8} {'Avg dB':>8} {'Clip':>6} {'Silence%':>9} {'Issues':>7} {'Usability':<18}")
print("-" * 88)

for r in all_reports:
    q = r.audio_quality
    print(
        f"{r.file_name:<25} "
        f"{r.duration_seconds:>7.1f}s "
        f"{str(q.avg_volume_db) + ' dB':>8} "
        f"{'✅' if not q.clipping_detected else '❌':>6} "
        f"{q.silence_ratio * 100:>8.1f}% "
        f"{len(r.issues):>7} "
        f"{r.overall_usability:<18}"
    )

print()
print("=" * 60)
print("💬 LLM SUMMARY PER FILE")
print("=" * 60)
for r in all_reports:
    print(f"\n🎵 {r.file_name}")
    print(f"   Summary    : {r.llm_summary}")
    print(f"   Usability  : {r.overall_usability.upper()}")
    if r.issues:
        print(f"   Issues     :")
        for issue in r.issues:
            print(f"     ⚠️  {issue}")
    print(f"   Recommendations:")
    for rec in r.recommendations:
        print(f"     → {rec}")



📊 ANALYSIS RESULTS COMPARISON
File                      Duration   Avg dB   Clip  Silence%  Issues Usability         
----------------------------------------------------------------------------------------
bad_audio.mp3               121.1s -16.6 dB      ❌     13.4%       3 unusable          
moonlight-plaza.mp3         854.5s -25.3 dB      ✅      1.2%       1 partially_usable  

💬 LLM SUMMARY PER FILE

🎵 bad_audio.mp3
   Summary    : The audio file bad_audio.mp3 has been deemed unusable by the Acoustic Expert due to clipping and high noise levels, which may cause distortion and reduce transcription accuracy. Although the Linguistic Expert did not detect any linguistic issues, the acoustic problems are significant and should be addressed. The file's low bitrate of 47.235 kbps may also affect audio quality. The silence ratio, however, is below the 20% threshold, contradicting the Acoustic Expert's initial claim of it exceeding 20%.
   Usability  : UNUSABLE
   Issues     :
     ⚠️  Aud

## 11. Batch Processing — Full Folder (Optional)

In [12]:
def batch_analyze(audio_dir: str, extensions: List[str] = [".wav", ".mp3", ".m4a"]) -> List[dict]:
    """
    Process multiple audio files from a directory.
    Returns aggregated insights.
    """
    audio_dir = Path(audio_dir)
    audio_files = []
    for ext in extensions:
        audio_files.extend(audio_dir.glob(f"*{ext}"))

    if not audio_files:
        print(f"❌ No audio files found in {audio_dir}")
        return []

    print(f"\n🎙️ Batch Processing: {len(audio_files)} files found")
    print("=" * 50)

    reports = []
    for i, file_path in enumerate(audio_files, 1):
        print(f"\n[{i}/{len(audio_files)}] Processing: {file_path.name}")
        try:
            report = analyze_audio(str(file_path))
            reports.append(report.model_dump())
        except Exception as e:
            print(f"❌ Failed to process {file_path.name}: {e}")

    # Aggregate insights
    print("\n" + "=" * 50)
    print("📊 BATCH SUMMARY")
    print("=" * 50)

    usability_counts = {}
    for r in reports:
        u = r.get("overall_usability", "unknown")
        usability_counts[u] = usability_counts.get(u, 0) + 1

    print(f"Total files processed: {len(reports)}")
    for status, count in usability_counts.items():
        print(f"  {status}: {count} files")

    return reports


# Uncomment to run batch from folder:
# batch_reports = batch_analyze(".")
# print(json.dumps(batch_reports, indent=2))

print("✅ Batch processor: batch_analyze() ready")
print("   → Uncomment last line to run batch from folder")

✅ Batch processor: batch_analyze() ready
   → Uncomment last line to run batch from folder


## 12. Save Report to JSON File

In [13]:
def save_report(report: AudioAnalysisReport, output_dir: str = "./reports"):
    """
    Save analysis report to JSON file.
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    file_name = report.file_name.replace(".", "_")
    output_file = output_path / f"report_{file_name}.json"

    with open(output_file, "w") as f:
        json.dump(report.model_dump(), f, indent=2)

    print(f"✅ Report saved: {output_file}")
    return str(output_file)


# Auto-save all analyzed reports
for r in all_reports:
    save_report(r)

print("✅ Save utility ready")

✅ Report saved: reports/report_bad_audio_mp3.json
✅ Report saved: reports/report_moonlight-plaza_mp3.json
✅ Save utility ready


## 12. Example Output

Real output from the two test files included in this repository.

### `bad_audio.mp3` — Unusable

```json
{
  "file_name": "bad_audio.mp3",
  "duration_seconds": 121.11,
  "audio_quality": {
    "silence_ratio": 0.1341,
    "clipping_detected": true,
    "avg_volume_db": -16.6
  },
  "issues": [
    "Audio clipping detected — may cause distortion",
    "High background noise detected — may reduce transcription accuracy",
    "Low bitrate: 47 kbps — may affect audio quality"
  ],
  "llm_summary": "The audio file bad_audio.mp3 is deemed unusable due to significant acoustic issues, including clipping at -0.4 dB, high noise levels, and a bitrate of 47 kbps below the recommended 64 kbps threshold. These combined issues cause distortion and reduce transcription accuracy.",
  "recommendations": [
    "Increase the bitrate to at least 64 kbps to improve audio quality.",
    "Apply noise reduction techniques to minimise high background noise.",
    "Re-record the audio to avoid clipping and ensure a higher quality signal.",
    "Consider using a lossless codec to preserve original audio quality.",
    "Implement audio normalisation to ensure consistent volume levels."
  ],
  "overall_usability": "unusable",
  "detected_languages": ["en"]
}
```

### `moonlight-plaza.mp3` — Partially Usable

```json
{
  "file_name": "moonlight-plaza.mp3",
  "duration_seconds": 854.53,
  "audio_quality": {
    "silence_ratio": 0.0116,
    "clipping_detected": false,
    "avg_volume_db": -25.3
  },
  "issues": [
    "Medium background noise detected — may reduce transcription accuracy"
  ],
  "llm_summary": "The audio file moonlight-plaza.mp3 is partially usable. Its bitrate of 128 kbps and silence ratio of 1.16% are within acceptable ranges, and no clipping was detected. However, medium background noise may reduce transcription accuracy and should be addressed before use in legal proceedings.",
  "recommendations": [
    "Apply noise reduction to minimise medium background noise before transcription.",
    "Verify transcription output for errors caused by background noise.",
    "Consider re-recording in a quieter environment for optimal quality."
  ],
  "overall_usability": "partially_usable",
  "detected_languages": ["en"]
}
```

## Architecture Notes

### Design Decisions

1. **Multi-agent with tiered models** — Agent 1 & 2 use `llama-3.1-8b-instant` (~12× cheaper) for structured JSON tasks; Agent 3 uses `llama-3.3-70b-versatile` for final synthesis
2. **Whisper for language detection** — Tool 4 samples 30s of audio via Whisper `tiny` to detect spoken language from the actual signal, not filename guesses
3. **LLM as interpreter, not calculator** — ffmpeg handles all numeric analysis; LLM only interprets results
4. **Python sanitiser** — drops LLM-generated issues that contradict raw measured numbers (math hallucination guard)
5. **Fault tolerance via Tenacity** — retry with exponential backoff on RateLimitError/Timeout; per-agent static fallback if all retries fail
6. **Pydantic for structured output** — consistent JSON schema regardless of LLM response variation

### Extensibility

- **New ffmpeg tools** — add a `detect_*()` function and wire into Step 4
- **New LLM agents** — add a specialist (e.g. speaker diarisation) between Agent 2 and Agent 3
- **MCP Server** — expose tools via FastMCP for agent integration
- **Batch at scale** — replace sequential calls with `asyncio.gather()` for parallel processing
- **OpenAI fallback** — swap `ChatGroq` with `ChatOpenAI` by changing one import

### Requirements

```
ffmpeg >= 4.0          (brew install ffmpeg)
python >= 3.9
openai-whisper         (Whisper tiny model ~75MB, downloaded on first run)
langchain
langchain-groq
pydantic >= 2.0
python-dotenv
groq
tenacity
httpx
```
